# Positions

In [ ]:
#| default_exp position

In [ ]:
#| export

from dataclasses import dataclass
from typing import Tuple, Dict, Optional
from sugar.helpers import normalize_address, ADDRESS_ZERO
from sugar.pool import LiquidityPool

In [ ]:
#| export

@dataclass(frozen=True)
class Position:
    """User's position in a pool, returned by `chain.get_positions(owner)`.

    Mirrors Sugar's `positions(limit, offset, account)` reader output. Amount
    fields are raw token amounts (wei-level uints) — for USD display, combine
    with `chain.get_prices(...)` and the pool's `token0/1`. For CL, `id` is the
    NFT tokenId; for basic pools `id` is 0 and tick / sqrt fields are unused.
    """
    chain_id: str
    chain_name: str
    # NFT tokenId for CL; 0 for basic
    id: int
    pool: LiquidityPool
    # unstaked liquidity (LP tokens for basic, V3 L for CL)
    liquidity: int
    # gauge-staked liquidity
    staked: int
    # raw token amounts (wei-level uints) at the pool's current price
    amount_token0: int
    amount_token1: int
    staked_token0: int
    staked_token1: int
    # accrued, unclaimed fees
    unstaked_earned0: int
    unstaked_earned1: int
    # accrued, unclaimed gauge emissions (in `pool.emissions_token` units)
    emissions_earned: int
    # CL tick bounds (0 for basic)
    tick_lower: int
    tick_upper: int
    sqrt_ratio_lower: int
    sqrt_ratio_upper: int
    # ALM contract managing this position; ADDRESS_ZERO if not ALM-managed
    alm: str

    @property
    def is_cl(self) -> bool: return self.pool.is_cl

    @property
    def is_alm(self) -> bool: return self.alm != ADDRESS_ZERO

    @property
    def is_in_range(self) -> bool:
        """For CL positions, True iff the pool's current tick is within [tick_lower, tick_upper).
        Always False for basic pools (the concept doesn't apply)."""
        return self.pool.is_cl and self.tick_lower <= self.pool.tick < self.tick_upper

    @classmethod
    def from_tuple(cls, t: Tuple, pools: Dict[str, LiquidityPool],
                   chain_id: str, chain_name: str) -> Optional["Position"]:
        # Sugar positions() tuple shape:
        # 0  id              uint256        # NFT tokenId (CL) or 0 (basic)
        # 1  lp              address
        # 2  liquidity       uint256
        # 3  staked          uint256
        # 4  amount0         uint256
        # 5  amount1         uint256
        # 6  staked0         uint256
        # 7  staked1         uint256
        # 8  unstaked_earned0  uint256
        # 9  unstaked_earned1  uint256
        # 10 emissions_earned  uint256
        # 11 tick_lower      int24
        # 12 tick_upper      int24
        # 13 sqrt_ratio_lower  uint160
        # 14 sqrt_ratio_upper  uint160
        # 15 locker          address
        # 16 unlocks_at      uint32
        # 17 alm             address
        lp = normalize_address(t[1])
        pool = pools.get(lp)
        if pool is None: return None
        return Position(
            chain_id=chain_id, chain_name=chain_name,
            id=t[0], pool=pool,
            liquidity=t[2], staked=t[3],
            amount_token0=t[4], amount_token1=t[5],
            staked_token0=t[6], staked_token1=t[7],
            unstaked_earned0=t[8], unstaked_earned1=t[9],
            emissions_earned=t[10],
            tick_lower=t[11], tick_upper=t[12],
            sqrt_ratio_lower=t[13], sqrt_ratio_upper=t[14],
            alm=normalize_address(t[17]),
        )

In [ ]:
# Position.from_tuple sanity — verifies field-index mapping against Sugar's tuple shape.
from types import SimpleNamespace
from fastcore.test import test_eq

cl_pool = SimpleNamespace(is_cl=True, tick=50)
basic_pool = SimpleNamespace(is_cl=False, tick=0)
lp_cl = normalize_address("0x1111111111111111111111111111111111111111")
lp_basic = normalize_address("0x2222222222222222222222222222222222222222")
alm_addr = normalize_address("0x3333333333333333333333333333333333333333")
pools = {lp_cl: cl_pool, lp_basic: basic_pool}

# CL position tuple
t_cl = (42, lp_cl, 1000, 500, 100, 200, 50, 75, 10, 20, 5, -100, 100, 12345, 67890, alm_addr)
p = Position.from_tuple(t_cl, pools, "10", "Optimism")
test_eq((p.id, p.liquidity, p.staked), (42, 1000, 500))
test_eq((p.amount_token0, p.amount_token1), (100, 200))
test_eq((p.staked_token0, p.staked_token1), (50, 75))
test_eq((p.unstaked_earned0, p.unstaked_earned1, p.emissions_earned), (10, 20, 5))
test_eq((p.tick_lower, p.tick_upper), (-100, 100))
test_eq((p.sqrt_ratio_lower, p.sqrt_ratio_upper), (12345, 67890))
test_eq(p.alm, alm_addr)
test_eq(p.is_cl, True)
test_eq(p.is_alm, True)
test_eq(p.is_in_range, True)  # cl_pool.tick=50 is within [-100, 100)

# in-range edge cases
out_low = SimpleNamespace(is_cl=True, tick=-101)
out_high = SimpleNamespace(is_cl=True, tick=100)  # upper is exclusive
in_low = SimpleNamespace(is_cl=True, tick=-100)   # lower is inclusive
for pool, expected in [(out_low, False), (out_high, False), (in_low, True)]:
    p_test = Position.from_tuple(t_cl, {lp_cl: pool}, "10", "Optimism")
    test_eq(p_test.is_in_range, expected)

# basic position tuple — id=0, ticks=0, alm=ADDRESS_ZERO
t_basic = (0, lp_basic, 999, 0, 1000, 2000, 0, 0, 5, 8, 0, 0, 0, 0, 0, ADDRESS_ZERO)
p = Position.from_tuple(t_basic, pools, "10", "Optimism")
test_eq((p.id, p.liquidity), (0, 999))
test_eq(p.is_cl, False)
test_eq(p.is_alm, False)
test_eq(p.is_in_range, False)  # basic is never in-range

# missing pool -> None (filtered by prepare_positions)
unknown_lp = normalize_address("0x9999999999999999999999999999999999999999")
t_unknown = (1, unknown_lp, 100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ADDRESS_ZERO)
test_eq(Position.from_tuple(t_unknown, pools, "10", "Optimism"), None)

In [ ]:
#| hide

import nbdev; nbdev.nbdev_export()